# Pre-Training Phase 2: High-Quality Data

**Paper §2.3** — at the 94 % point of training the data mixture shifts to
high-quality sources such as Wikipedia, curated synthetics, and premium web text.
This final 6 % (≈ 1.5 T tokens) sharpens the model's knowledge and reduces noise.

Resumes automatically from the latest Phase 1 checkpoint in `PRETRAIN_CKPT_DIR`.

### Notebook contents
1. Environment setup
2. Imports & hyperparameters
3. Tokenizer
4. Dataset
5. Model (load Phase 1 checkpoint)
6. Optimizer
7. Checkpoint manager
8. Training loop
9. Loss curve & evaluation


## 0. Environment Setup

Detects Colab / Kaggle, installs packages, and adds the repo to `sys.path`.

In [ ]:
import os, sys

IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


In [ ]:
if IN_COLAB or IN_KAGGLE:
    # Set ACCELERATOR to "cuda12" (GPU) or "tpu" (Colab TPU only).
    ACCELERATOR = "cuda12"
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         f"jax[{ACCELERATOR}]", "flax", "optax", "orbax-checkpoint",
         "datasets", "transformers", "matplotlib"],
        check=True,
    )


In [ ]:
import pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"


def _detect_repo_root() -> pathlib.Path:
    """Detect local repo root by searching upward for key project files."""
    cwd = pathlib.Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "training_shared.py").exists() and (candidate / "nemotron.py").exists():
            return candidate
    return cwd


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    REPO_DIR = _detect_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("REPO_DIR:", REPO_DIR)


## 1. Imports & Hyperparameters

In [ ]:
import math
import pathlib

import jax
import jax.numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp
from datasets import load_dataset
from flax import nnx
from transformers import AutoTokenizer

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock
from training_shared import (
    VOCAB_SIZE, CHUNK_SIZE,
    PRETRAIN_SEQ_LEN, PRETRAIN_BATCH, PHASE2_STEPS,
    PRETRAIN_PEAK_LR, PRETRAIN_MIN_LR,
    PRETRAIN_WD, PRETRAIN_B1, PRETRAIN_B2,
    PRETRAIN_CKPT_DIR, PRETRAIN_CKPT_EVERY, PRETRAIN_VAL_STEPS,
    build_model, collect_moe_layers, make_decayed_lr_optimizer,
    load_pretrain_data, make_batches, pretrain_step, update_moe_biases,
    evaluate_pretrain, make_checkpoint_manager, save_checkpoint,
    try_load_from_dir,
)

print("JAX devices:", jax.devices())


In [ ]:
SEQ_LEN     = PRETRAIN_SEQ_LEN
BATCH_SIZE  = PRETRAIN_BATCH
TRAIN_STEPS = PHASE2_STEPS       # 500
CKPT_DIR    = PRETRAIN_CKPT_DIR

# Phase 2 has a shorter WSD schedule re-initialised for its own budget.
phase2_warmup = max(1, TRAIN_STEPS // 10)
phase2_stable = max(1, TRAIN_STEPS // 2)
phase2_decay  = max(1, TRAIN_STEPS - phase2_warmup - phase2_stable)

print(f"Steps={TRAIN_STEPS} | warmup={phase2_warmup} | stable={phase2_stable} | decay={phase2_decay}")


## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {tokenizer.vocab_size}")


## 3. Dataset

Phase 2 samples from a different (non-overlapping) region of the dataset to simulate the high-quality data mixture.

In [ ]:
train_chunks = load_pretrain_data(
    split="train", max_samples=100, seq_len=SEQ_LEN, tokenizer=tokenizer, skip=500
)
val_chunks = load_pretrain_data(
    split="train", max_samples=30,  seq_len=SEQ_LEN, tokenizer=tokenizer, skip=600
)

print(f"Train chunks : {len(train_chunks)}  shape={train_chunks.shape}")
print(f"Val   chunks : {len(val_chunks)}   shape={val_chunks.shape}")


## 4. Model — Load Phase 1 Checkpoint

In [ ]:
print("Building model ...")
model = build_model(seed=0)
moe_layers = collect_moe_layers(model)

loaded = try_load_from_dir(PRETRAIN_CKPT_DIR, model, model.config)
if loaded:
    print("Resumed from Phase 1 checkpoint.")
else:
    print("No Phase 1 checkpoint found — starting from scratch.")


## 5. Optimizer

In [ ]:
tx = make_decayed_lr_optimizer(
    peak_lr=PRETRAIN_PEAK_LR,
    min_lr=PRETRAIN_MIN_LR,
    warmup_steps=phase2_warmup,
    stable_steps=phase2_stable,
    decay_steps=phase2_decay,
    weight_decay=PRETRAIN_WD,
    b1=PRETRAIN_B1,
    b2=PRETRAIN_B2,
)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)
print("Optimizer created.")


## 6. Checkpoint Manager

In [ ]:
ckpt_mgr = make_checkpoint_manager(CKPT_DIR)
print(f"Checkpoints → {CKPT_DIR}  (every {PRETRAIN_CKPT_EVERY} steps)")


## 7. Training Loop

In [ ]:
print(f"\n=== Pre-Training Phase 2: High-Quality Data ===")
print(f"Training for {TRAIN_STEPS} steps ...\n")

loss_history: list[float] = []
step = 0
batch_iter = iter(make_batches(train_chunks, BATCH_SIZE))

while step < TRAIN_STEPS:
    try:
        batch_np = next(batch_iter)
    except StopIteration:
        batch_iter = iter(make_batches(train_chunks, BATCH_SIZE))
        batch_np = next(batch_iter)

    batch = jnp.array(batch_np)
    loss  = pretrain_step(model, optimizer, batch)
    update_moe_biases(moe_layers)
    step += 1
    loss_history.append(float(loss))

    if step % 100 == 0:
        val_loss, ppl = evaluate_pretrain(model, val_chunks, PRETRAIN_VAL_STEPS, moe_layers)
        print(f"  Step {step:4d} | train_loss={float(loss):.4f} | "
              f"val_loss={val_loss:.4f} | ppl={ppl:.1f}")

    if step % PRETRAIN_CKPT_EVERY == 0:
        save_checkpoint(ckpt_mgr, model, step)

save_checkpoint(ckpt_mgr, model, step)
print("\nPhase 2 complete.")


## 8. Loss Curve & Evaluation

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(loss_history) + 1), loss_history)
    plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("Phase 2 Training Loss")
    plt.grid(True); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")


In [ ]:
val_loss, ppl = evaluate_pretrain(model, val_chunks, PRETRAIN_VAL_STEPS, moe_layers)
print(f"Final val_loss={val_loss:.4f}  |  perplexity={ppl:.1f}")
